<a href="https://colab.research.google.com/github/taek20230541/maritimedatamining/blob/main/Colab_%EC%8B%9C%EC%9E%91%ED%95%98%EA%B8%B0%EC%9D%98_%EC%82%AC%EB%B3%B8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 문제 9

In [ ]:
import pandas as pd

# 1. 데이터 로드
url = "https://raw.githubusercontent.com/lovedlim/bigdata_analyst_cert/main/part4/ch7/air_quality.csv"
df = pd.read_csv(url)

# 2. IQR(Interquartile Range) 계산
Q1 = df['CO2'].quantile(0.25)
Q3 = df['CO2'].quantile(0.75)
IQR = Q3 - Q1

# 3. 이상치 경계값(Threshold) 설정
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

# 4. 이상치 식별 (OR 연산자 | 활용)
outliers = df[(df['CO2'] < lower_bound) | (df['CO2'] > upper_bound)]

# 5. 결과 출력
print(f"하한선: {lower_bound:.2f}, 상한선: {upper_bound:.2f}")
print(f"이상치 개수: {len(outliers)}")
print("-" * 30)
print(outliers.head()) # 이상치 데이터 상단 확인

하한선: 331.46, 상한선: 471.17
이상치 개수: 304
------------------------------
                    Date        CO         CO2         O2
88   2024-01-04 16:00:00  0.447024  603.471090  20.712496
155  2024-01-07 11:00:00  0.428565  588.211313  22.013521
185  2024-01-08 17:00:00  0.571400  472.852730  21.667225
196  2024-01-09 04:00:00  0.411614  607.631613  21.072704
228  2024-01-10 12:00:00  0.428470  595.429242  20.319365


# 문제 10


In [ ]:
import pandas as pd

# 1. 데이터 로드 (keep_default_na=False 옵션으로 'NA' 대륙 인식 오류 방지)
url = "https://raw.githubusercontent.com/lovedlim/bigdata_analyst_cert/main/part4/ch8/drinks.csv"
df = pd.read_csv(url, keep_default_na=False)

# 2. 대륙별 맥주 소비량 평균 계산 및 최다 소비 대륙(Label) 추출
# groupby 결과 중 beer_servings의 평균이 가장 높은 인덱스명을 가져옴
target_continent = df.groupby("continent")['beer_servings'].mean().idxmax()

# 3. 해당 대륙 데이터만 필터링하여 정렬
# 맥주 소비량 기준 내림차순(ascending=False) 정렬
res_df = df[df['continent'] == target_continent].sort_values('beer_servings', ascending=False)

# 4. 결과 추출 (해당 대륙 내 소비량 5위 국가의 값)
# iloc[4]는 순서상 5번째 행을 의미함
result = res_df.iloc[4]['beer_servings']

# 출력부
print(f"맥주 소비량이 가장 많은 대륙: {target_continent}")
print(f"해당 대륙 내 맥주 소비량 5위 국가의 값: {result}")

맥주 소비량이 가장 많은 대륙: Europe
해당 대륙 내 맥주 소비량 5위 국가의 값: 313


# 문제 11 (방법1)

In [ ]:
import pandas as pd

# 1. 데이터 로드
url = "https://raw.githubusercontent.com/lovedlim/bigdata_analyst_cert/main/part4/ch8/tourist.csv"
df = pd.read_csv(url)

# 2. 파생변수 생성 (방문객 합계 및 관광객 비율)
# 합계 계산 시 컬럼명을 리스트로 묶어 .sum(axis=1)을 쓰면 더 깔끔합니다.
cols = ['관광', '공무', '사업', '기타']
df['방문객합계'] = df[cols].sum(axis=1)
df['관광객비율'] = df['관광'] / df['방문객합계']

# 3. 조건별 데이터 추출
# (a) 관광객 비율 기준 내림차순 정렬 후, 2위(index 1)의 '사업' 방문객 수
df_ratio_sort = df.sort_values('관광객비율', ascending=False)
val_a = df_ratio_sort.iloc[1]['사업']

# (b) '관광' 방문객 수 기준 내림차순 정렬 후, 2위(index 1)의 '공무' 방문객 수
df_tour_sort = df.sort_values('관광', ascending=False)
val_b = df_tour_sort.iloc[1]['공무']

# 4. 최종 결과 출력
print(f"조건 (a)의 사업 방문객: {val_a}")
print(f"조건 (b)의 공무 방문객: {val_b}")
print(f"최종 합계(a+b): {val_a + val_b}")

조건 (a)의 사업 방문객: 203
조건 (b)의 공무 방문객: 238
최종 합계(a+b): 441


# 문제 11 (방법2)

In [ ]:
import pandas as pd

# 1. 데이터 로드
url = "https://raw.githubusercontent.com/lovedlim/bigdata_analyst_cert/main/part4/ch8/tourist.csv"
df = pd.read_csv(url)

# 2. 파생변수 생성 (방문객 합계 및 관광객 비율)
# 합계 계산 시 컬럼 리스트를 활용하면 오타 위험이 줄어듭니다.
cols = ['관광', '공무', '사업', '기타']
df['방문객합계'] = df[cols].sum(axis=1)
df['관광객비율'] = df['관광'] / df['방문객합계']

# 3. nlargest를 이용한 상위 2번째 값 추출
# nlargest(n, '컬럼명'): 해당 컬럼 기준 큰 순서대로 n개 추출
# .iloc[1]: 추출된 상위 2개 중 2번째(인덱스 1) 행 선택

# (a) 관광객 비율 상위 2위 국가의 '사업' 방문객 수
val_a = df.nlargest(2, '관광객비율').iloc[1]['사업']

# (b) '관광' 방문객 수 상위 2위 국가의 '공무' 방문객 수
val_b = df.nlargest(2, '관광').iloc[1]['공무']

# 4. 결과 출력
print(f"조건 a(사업): {val_a}")
print(f"조건 b(공무): {val_b}")
print(f"최종 결과(a + b): {val_a + val_b}")

조건 a(사업): 203
조건 b(공무): 238
최종 결과(a + b): 441


# 문제 12 (방법1)

In [ ]:
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

# 1. 데이터 로드
url = "https://raw.githubusercontent.com/lovedlim/bigdata_analyst_cert/main/part4/ch8/chem.csv"
df = pd.read_csv(url)

# 2. Min-Max 스케일링 (효율적인 일괄 처리)
# 각 컬럼을 따로 fit_transform하기보다 리스트로 묶어 한 번에 처리하는 것이 효율적입니다.
scaler = MinMaxScaler()
cols_to_scale = ['co', 'nmhc']
df[[c + '_scaled' for c in cols_to_scale]] = scaler.fit_transform(df[cols_to_scale])

# 3. 표준편차(Standard Deviation) 계산
# 판다스의 std()는 기본적으로 비편향 표준편차(ddof=1)를 계산합니다.
co_std = df['co_scaled'].std()
nmhc_std = df['nmhc_scaled'].std()

# 4. 차이 계산 및 반올림 (절대값 필요 시 abs() 활용 가능)
# 문제에서 단순히 (co - nmhc)를 요구했다면 그대로 진행합니다.
std_diff = round(co_std - nmhc_std, 3)

# 결과 출력
print(f"CO 스케일링 후 표준편차: {co_std:.4f}")
print(f"NMHC 스케일링 후 표준편차: {nmhc_std:.4f}")
print("-" * 30)
print(f"표준편차 차이(반올림): {std_diff}")

CO 스케일링 후 표준편차: 0.2857
NMHC 스케일링 후 표준편차: 0.3031
------------------------------
표준편차 차이(반올림): -0.017


# 문제 12 (방법2)


In [ ]:
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

# 1. 데이터 로드
url = "https://raw.githubusercontent.com/lovedlim/bigdata_analyst_cert/main/part4/ch8/chem.csv"
df = pd.read_csv(url)

# 2. Min-Max 스케일링 수행 (2차원 데이터 입력 필요)
scaler = MinMaxScaler()
co_scaled = scaler.fit_transform(df[['co']])
nmhc_scaled = scaler.fit_transform(df[['nmhc']])

# 3. 표준편차 계산 (넘파이 방식: ddof=0)
co_std = co_scaled.std()
nmhc_std = nmhc_scaled.std()

# 4. 표준편차 차이 계산 및 반올림
std_diff = round(co_std - nmhc_std, 3)

# --------------------------------------------------
# 최종 결과 출력 (요청하신 2가지 답안)
# --------------------------------------------------
# 결과 1) 각각의 표준편차
print(f"1) 각 컬럼의 표준편차: co={co_std:.4f}, nmhc={nmhc_std:.4f}")

# 결과 2) 표준편차 차이 및 반올림 값
print(f"2) 표준편차 차이(반올림): {std_diff}")

1) 각 컬럼의 표준편차: co=0.2842, nmhc=0.3015
2) 표준편차 차이(반올림): -0.017


# 문제 13 (방법1)

In [ ]:
import pandas as pd

# 1. 데이터 로드
url = "https://raw.githubusercontent.com/lovedlim/bigdata_analyst_cert_v2/main/part4/ch9/loan.csv"
df = pd.read_csv(url)

# 2. 파생변수 생성 (총대출액)
# 신용대출과 담보대출을 합산하여 새로운 컬럼을 만듭니다.
df['총대출액'] = df['신용대출'] + df['담보대출']

# 3. 지역코드와 성별에 따른 총대출액 합계 계산 후 재구조화
# unstack()을 통해 '성별' 인덱스를 컬럼(열)으로 올립니다.
grouped = df.groupby(['지역코드', '성별'])['총대출액'].sum().unstack()

# 4. 성별 간 대출액 차이 계산 (절대값 활용)
# 컬럼명이 1(남성), 2(여성)인 경우를 가정하여 차이를 계산합니다.
grouped['차이'] = abs(grouped[1] - grouped[2])

# 5. 차이가 가장 큰 지역코드(Index) 추출
result = grouped['차이'].idxmax()

# 최종 결과 출력
print(f"성별 대출액 차이가 가장 큰 지역코드: {result}")
print("-" * 35)
print(grouped.sort_values('차이', ascending=False).head()) # 상위 데이터 확인

성별 대출액 차이가 가장 큰 지역코드: 4100000278
-----------------------------------
성별                 1         2        차이
지역코드                                    
4100000278  72934984  12605775  60329209
4100000089   8802541  66042097  57239556
4100000147   3493195  58151505  54658310
4100000061  62875894   9794865  53081029
4100000109   3153781  54825674  51671893


# 문제 13 (방법2)

In [ ]:
import pandas as pd

# 1. 데이터 로드
url = "https://raw.githubusercontent.com/lovedlim/bigdata_analyst_cert_v2/main/part4/ch9/loan.csv"
df = pd.read_csv(url)

# 2. 파생변수 생성 (총대출액)
# 개별 대출 항목을 합산하여 분석 대상 컬럼을 만듭니다.
df['총대출액'] = df['신용대출'] + df['담보대출']

# 3. 데이터 재구조화 (Pivot Table 생성)
# 행(index)은 지역코드, 열(columns)은 성별로 설정하여 총대출액의 합계(sum)를 집계합니다.
pivot_result = df.pivot_table(index='지역코드',
                              columns='성별',
                              values='총대출액',
                              aggfunc='sum')

# 4. 성별 간 총대출액 차이 계산 (절대값 활용)
# 성별 코드 1(남성)과 2(여성)의 대출액 차이를 구합니다.
pivot_result['차이'] = abs(pivot_result[1] - pivot_result[2])

# 5. 차이가 가장 큰 지역코드(Index) 추출
result = pivot_result['차이'].idxmax()

# 최종 결과 출력
print(f"성별 대출액 차이가 가장 큰 지역코드: {result}")
print("-" * 40)
print(pivot_result.sort_values('차이', ascending=False).head()) # 상위 결과 확인

성별 대출액 차이가 가장 큰 지역코드: 4100000278
----------------------------------------
성별                 1         2        차이
지역코드                                    
4100000278  72934984  12605775  60329209
4100000089   8802541  66042097  57239556
4100000147   3493195  58151505  54658310
4100000061  62875894   9794865  53081029
4100000109   3153781  54825674  51671893


# 문제 14 (방법1)

In [ ]:
import pandas as pd

# 1. 데이터 로드
url = "https://raw.githubusercontent.com/lovedlim/bigdata_analyst_cert_v2/main/part4/ch9/crime.csv"
df = pd.read_csv(url)

# 2. 발생건수와 검거건수 분리 및 인덱스 초기화
# [:, 2:] 슬라이싱을 통해 '연도'와 '구분' 컬럼을 제외한 수치 데이터만 추출합니다.
df_occurrence = df[df["구분"] == "발생건수"].iloc[:, 2:].reset_index(drop=True)
df_arrest = df[df["구분"] == "검거건수"].iloc[:, 2:].reset_index(drop=True)

# 3. 검거율 계산 (검거건수 / 발생건수)
# 두 데이터프레임의 구조가 동일하므로 바로 나눗셈 연산이 가능합니다.
df_rate = df_arrest / df_occurrence

# 4. 연도별(행별) 검거율이 가장 높은 범죄유형(컬럼명) 찾기
# idxmax(axis=1)은 각 행에서 최댓값을 가진 열의 이름을 반환합니다.
top_crimes = df_rate.idxmax(axis=1)

# 5. 해당 범죄유형의 검거건수 합산
total_arrests = 0
for idx, crime_type in enumerate(top_crimes):
    # 각 연도(idx)에서 특정 범죄유형(crime_type)의 검거건수를 더합니다.
    total_arrests += df_arrest.loc[idx, crime_type]

# 최종 결과 출력
print(f"연도별 최고 검거율 범죄의 총 검거건수 합계: {total_arrests}")

연도별 최고 검거율 범죄의 총 검거건수 합계: 7799


# 문제 14 (방법2)

In [ ]:
import pandas as pd

# 1. 데이터 로드
url = "https://raw.githubusercontent.com/lovedlim/bigdata_analyst_cert_v2/main/part4/ch9/crime.csv"
df = pd.read_csv(url)

# 2. 데이터 재구조화 (Wide to Long)
# 범죄유형 컬럼들을 '범죄유형'이라는 하나의 열로 모읍니다.
df_long = pd.melt(df, id_vars=["연도", "구분"], var_name="범죄유형", value_name="건수")

# 3. 피벗 테이블 생성 (Long to Wide)
# '구분'(발생건수, 검거건수)을 컬럼으로 올려서 한 행에서 연산이 가능하게 만듭니다.
df_pivot = df_long.pivot_table(index=["연도", "범죄유형"],
                               columns="구분",
                               values="건수").reset_index()

# 4. 검거율 계산 및 최고 검거율 행 식별
df_pivot["검거율"] = df_pivot["검거건수"] / df_pivot["발생건수"]

# 각 연도별로 검거율이 가장 높은 인덱스(idxmax) 추출
best_indices = df_pivot.groupby("연도")["검거율"].idxmax()

# 5. 해당 행들의 검거건수 합계 계산
# 추출된 인덱스를 활용해 검거건수만 필터링하여 합산합니다.
result = df_pivot.loc[best_indices, "검거건수"].sum()

# 최종 결과 출력
print(f"연도별 최고 검거율 범죄의 검거건수 총합: {int(result)}")

연도별 최고 검거율 범죄의 검거건수 총합: 7799


# 문제 15 (방법1)

In [ ]:
import pandas as pd

# 1. 데이터 로드
url = "https://raw.githubusercontent.com/lovedlim/bigdata_analyst_cert_v2/main/part4/ch9/hr.csv"
df = pd.read_csv(url)

# 2. 결측치 처리 (Imputation)
# 1) 만족도: 전체 평균으로 채움
df["만족도"] = df["만족도"].fillna(df["만족도"].mean())

# 2) 근속연수: 부서 및 성과등급별 평균으로 채움 (데이터 왜곡 최소화)
# transform은 그룹별 계산값을 원본 행 위치에 맞춰 반환합니다.
group_mean = df.groupby(["부서", "성과등급"])["근속연수"].transform("mean").astype(int)
df["근속연수"] = df["근속연수"].fillna(group_mean)

# 3. 파생 변수 생성 및 특정 순위 값 추출
# (A) 연봉/근속연수 기준 3위의 '근속연수'
df["연봉_근속연수"] = df["연봉"] / df["근속연수"]
# nlargest(3, ...)로 상위 3명을 뽑고, iloc[-1]로 그 중 마지막(3위)을 선택합니다.
top3_year_efficiency = df.nlargest(3, "연봉_근속연수")
A = top3_year_efficiency.iloc[-1]["근속연수"]

# (B) 연봉/만족도 기준 2위의 '교육참가횟수'
df["연봉_만족도"] = df["연봉"] / df["만족도"]
# nlargest(2, ...)로 상위 2명을 뽑고, iloc[-1]로 그 중 마지막(2위)을 선택합니다.
top2_satisfaction_efficiency = df.nlargest(2, "연봉_만족도")
B = top2_satisfaction_efficiency.iloc[-1]["교육참가횟수"]

# 4. 결과 출력
print(f"세 번째로 높은 '연봉/근속연수'의 근속연수 (A): {A}")
print(f"두 번째로 높은 '연봉/만족도'의 교육참가횟수 (B): {B}")
print(f"최종 결과 (A + B): {A + B}")

세 번째로 높은 '연봉/근속연수'의 근속연수 (A): 1.0
두 번째로 높은 '연봉/만족도'의 교육참가횟수 (B): 6
최종 결과 (A + B): 7.0


# 문제 15 (방법2)

In [ ]:
import pandas as pd

# 1. 데이터 로드
url = "https://raw.githubusercontent.com/lovedlim/bigdata_analyst_cert_v2/main/part4/ch9/hr.csv"
df = pd.read_csv(url)

# 2. 결측치 처리 (Imputation)
# 1) 만족도: 전체 평균으로 채움
df["만족도"] = df["만족도"].fillna(df["만족도"].mean())

# 2) 근속연수: 부서 및 성과등급별 평균으로 채움 (데이터 왜곡 최소화)
# transform은 그룹별 계산값을 원본 행 위치에 맞춰 반환하므로 fillna와 찰떡궁합입니다.
group_mean = df.groupby(["부서", "성과등급"])["근속연수"].transform("mean").astype(int)
df["근속연수"] = df["근속연수"].fillna(group_mean)

# 3. 파생 변수 생성 및 특정 순위 값 추출
# (A) 연봉/근속연수 기준 3위의 '근속연수'
df["연봉_근속연수"] = df["연봉"] / df["근속연수"]
# 내림차순 정렬 후 3번째(index 2) 데이터를 추출합니다.
df_sorted_year = df.sort_values(by="연봉_근속연수", ascending=False)
A = df_sorted_year.iloc[2]["근속연수"]

# (B) 연봉/만족도 기준 2위의 '교육참가횟수'
df["연봉_만족도"] = df["연봉"] / df["만족도"]
# 내림차순 정렬 후 2번째(index 1) 데이터를 추출합니다.
df_sorted_sat = df.sort_values(by="연봉_만족도", ascending=False)
B = df_sorted_sat.iloc[1]["교육참가횟수"]

# 4. 결과 출력
print(f"세 번째로 높은 '연봉/근속연수'의 근속연수 (A): {A}")
print(f"두 번째로 높은 '연봉/만족도'의 교육참가횟수 (B): {B}")
print(f"최종 합계 (A + B): {A + B}")

세 번째로 높은 '연봉/근속연수'의 근속연수 (A): 1.0
두 번째로 높은 '연봉/만족도'의 교육참가횟수 (B): 6
최종 합계 (A + B): 7.0
